# Tabular Foundation Models II -- TabICL

Notation follows the slides: a table has $n$ rows (samples) and $m$ columns (features); column-wise embeddings have dimension $d$; a row embedding is $h_i \in \mathbb{R}^{4d}$ (4 [CLS] tokens).

## TabICL: The Two-stage Embedding Design

TabICL's tabular embedding module has two stages applied in sequence: a _distribution-aware column-wise embedding_ ($\mathrm{TF}_{\textrm{col}}$) and a _context-aware row-wise interaction_ ($\mathrm{TF}_{\textrm{row}}$).

1. What does each stage take as input and what does it produce?

2. Which "distribution" does $\mathrm{TF}_{\textrm{col}}$ refer to?

3. Explain why the order is colummn-first, then row, does it matter?

4. Regarding the column-wise stage, why is a permutation-invariant set transformer the right choice for embedding a single column?

## Architecture

1. Briefly compare TabPFNv2 and TabICL in terms of (a) how each handles the row/column attention, and (b) the resulting computational complexity. Why does TabICL scale better to large $n$?

## Rotary Embeddings

> Background: the original Transformer (Vaswani et al., 2017) adds a fixed sinusoidal positional encoding to the input embeddings. TabICL instead uses rotary positional embeddings (RoPE). RoPE rotates the query and key vectors as a function of position.
> For a 2-dimensional sub-block of a vector $x$ at position $p$:
>
> $$R(p)\begin{bmatrix} x_{2i} \\ x_{2i+1}\end{bmatrix}
>   = \begin{bmatrix} \cos\theta_i & -\sin\theta_i \\ \sin\theta_i & \cos\theta_i \end{bmatrix}
>     \begin{bmatrix} x_{2i} \\ x_{2i+1}\end{bmatrix},
>   \qquad \theta_i = \frac{p}{100000^{\,2i/d}}$$
>
> and attention becomes
> $\text{Attention}(Q,K,V) = \text{softmax}\!\left(\dfrac{(R(p_Q)Q)\,(R(p_K)K)^\top}{\sqrt d}\right)V.$


1. Last week we saw that TabPFNv2 uses no positional encoding at all, and we treated this as a feature: table row/column order is arbitrary, so the model should be permutation-invariant. TabICL, by contrast, adds RoPE during the row-wise interaction stage. What problem does this solve, and what desirable property does it deliberately give up?

2. State some similarities and at least three concrete differences between sinusoidal and RoPE embeddings. 

3. Implement RoPE for one even-dimensional vector in NumPy.
Fill in:

In [ ]:
import numpy as np

def rope(x: np.ndarray, p: int, base: float = 100_000.0) -> np.ndarray:
    """Apply rotary positional embedding to a single vector.
     
    Args:
        x (np.ndarray): Single vector of even length `d`.
        p (int): position to compute the embedding at.
        base (float): Defaults to 100,000.
    
    Returns:
        the rotated vector of the same shape.
    """
    d = x.shape[0]
    # 1. compute theta_i for i = 0 .. d/2 - 1
    # 2. for each pair (x[2i], x[2i+1])
    #    apply the 2x2 rotation by theta_i * ... ?
    # 3. reassemble and return

4. For the vector $x = [1, 0, 1, 0]^\top$ at position $p=1$, compute $R(1)\,x$.

5. Your implementation should confirm the relative-position property by comparing `rope(q, m) @ rope(k, n)` for a few `(m, n)` pairs with the same offset

# Homework

1. Finish this exercise sheet and review the solutions! The sheets are not graded but solving them helps *a lot* to prepare for the exam. We provide solutions to the exercises on Ilias. Write your own solutions down FIRST (best by hand!) and review our solutions only AFTER you finished the exercises.
2. Read [scGPT, Cui et al. 2024](https://www.nature.com/articles/s41592-024-02201-0), a GPT-style foundation model in the single-cell domain.